In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.decomposition import PCA


<h4> Generate synthetic CSP feature matrices for two classes

In [ ]:
# generate synthetic data using params from the paper
# csp features would be computed by csp.transform(X) = w.T @ X, where X is eeg data. 
# Final dims of features will be 40 x 5 x 4 (see csp explanation for more details). 
# 
# ^ Here,however I took 40 x 24 instead of 40 x 20 to differentiate the num trials/2 and features.
# Later on we compute scatter matrices and eigenvectors, where the dimensionality will be important. 
# Having different number of trials for a class and features makes these operations more clear.
num_trials = 40
num_features = 24
np.random.seed(42)

# first 20 = left (class 0), last 20 = right (class 1)
X_left  = np.random.randn( num_trials // 2, num_features)
X_right = np.random.randn( num_trials // 2, num_features) + 0.8  # slight offset so classes differ

print("Shape synethetic features for class 0",X_left.shape)

<h4> LDA: Using the fisher criterion to maximize interclass mean distance and minimize within class variance. </h4>


$$J(w) = \frac{w^T S_B w}{w^T S_W w}$$

The goal is to find a $w$ such that this is achieved:

1. $S_W$ is within class scatter and $S_B$ is the between class scatter matrix.
2. We want to maximize numerator and minimize the denominator.

Refer to the math notes for more details on the notations and what they mean.


<h4> Compute mean, recenter data and visualize spread for features across classes

In [ ]:
mu_left = np.mean(X_left,axis=0)
mu_right = np.mean(X_right,axis=0)
print("Shape of mean computed",mu_left.shape)

# plot variation of features in mean centered data
X_left_cent  = X_left - mu_left
X_right_cent = X_right - mu_right

#plot spread of features in 2D using PCA
X_all = np.concatenate([X_left_cent, X_right_cent], axis=0)  # (40, 24)
labels = np.array([0]*20 + [1]*20)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_all)  # (40, 2)
print("Shape PCA",X_pca.shape)

# plot PC1 vs PC2 for both classes
plt.scatter(X_pca[labels==0,0],X_pca[labels==0,1],label='left')
plt.scatter(X_pca[labels==1,0],X_pca[labels==1,1],label='right')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title("PC1 vs PC2: left and right classes")
plt.legend()
plt.show()




<h4> Compute within class and between class scatter matrices

In [ ]:
# within class
Sw_left = (X_left_cent.T @ X_left_cent) 
Sw_right = (X_right_cent.T @ X_right_cent) 
Sw_final = Sw_left + Sw_right

# between class
mu_bw_class = (mu_left + mu_right) / 2
Sb_left     = (mu_left - mu_bw_class).reshape(-1,1)
Sb_right    = (mu_right - mu_bw_class).reshape(-1,1)
Sb_final    = (Sb_left @ Sb_left.T) + (Sb_right @ Sb_right.T)


print("Within class scatter matrix shape", Sw_final.shape)
print("Between class scatter matrix shape", Sb_final.shape)



<h4> Finding "W" using the general eigenvalue problem. </h4>

$S_W⁻¹ S_B w = λw$

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(np.linalg.inv(Sw_final) @ Sb_final)

# sort by largest eigenvalue
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

w = eigenvectors[:, 0].real  
print("LDA axis shape:", w.shape)

<h4> Project the data onto the eigenvector axis and plot LDA projection

In [ ]:
z_left  = X_left @ w 
z_right = X_right @ w 

# plot the features after projection into LDA space and compare it with previous PCA plot
fig,ax = plt.subplots(1,2, figsize=(12,4))
ax[0].scatter(X_pca[labels==0,0],X_pca[labels==0,1],label='left')
ax[0].scatter(X_pca[labels==1,0],X_pca[labels==1,1],label='right')
ax[0].set_xlabel('PC1')
ax[0].set_ylabel('PC2')
ax[0].set_title("Before: PC1 vs PC2, left and right classes")
ax[0].legend(loc='upper left')


ax[1].scatter(z_left,np.zeros((len(z_left),)), label='left')
ax[1].scatter(z_right,np.zeros((len(z_right),)), label='right')
ax[1].set_title('After: LDA projection')
ax[1].legend(loc='upper left')

plt.show()
